In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.5 — Training Enhancements: Kaiming Init, Label
# Smoothing, and Mixup (an ablation study)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY default layer initialisation is suboptimal for ReLU
#     networks, and what Kaiming (He) initialisation changes
#   - Explain label smoothing as "stop being so sure of yourself" —
#     soft targets that combat overconfidence
#   - Implement Mixup: training on convex combinations of image PAIRS
#     (vicinal risk minimisation, Zhang et al. 2018)
#   - Run a controlled ablation — change ONE lever at a time and track
#     every variant with ExperimentTracker
#   - Measure overconfidence honestly (mean max-softmax probability)
#     instead of asserting it
#   - Apply the levers to a limited-labelled-data agritech scenario
#
# PREREQUISITES: M5/ex_2/01_simple_cnn.py and 02_resnet_se.py (CNN
#   training on CIFAR-10, ExperimentTracker basics)
# ESTIMATED TIME: ~35 min
#
# DATASET: CIFAR-10. To keep the ablation affordable on CPU we train on
#   a 20,000-image subset of the 50K training images (disclosed, fixed
#   prefix after the seeded loader shuffle) and evaluate on the FULL
#   10K validation set. Four variants x 6 epochs — small enough to run
#   on a laptop, large enough to see each lever's effect.
#
# PHASES:
#   1. THEORY  — Three levers: initialisation, soft targets, vicinal data
#   2. BUILD   — kaiming_init, mixup_batch, mixup_criterion, AblationCNN
#   3. TRAIN   — 4-variant ablation tracked with ExperimentTracker
#   4. VISUALISE — Accuracy curves, Mixup composites, confidence shift
#   5. APPLY   — Limited-label crop-disease triage for an agritech startup
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Three Levers That Cost Nothing at Inference


THREE TRAINING-TIME LEVERS (zero inference cost):

  1. KAIMING INIT:  std = sqrt(2/fan_in) keeps signal variance alive
     through ReLU layers. Default init is tuned for linear nets.
  2. LABEL SMOOTHING: targets become 0.9 / 0.01 instead of 1.0 / 0.0.
     The model stops being rewarded for arbitrary confidence.
  3. MIXUP: train on lam*x_i + (1-lam)*x_j with the same convex mix
     of the losses. Forces linear behaviour BETWEEN examples.

  ABLATION DISCIPLINE: change ONE lever at a time, keep everything else
  identical (same architecture, same subset, same epochs, same seed),
  and let ExperimentTracker hold the receipts.


In [ ]:
# All three levers in this file share one property: they change TRAINING
# only. The deployed model is the same architecture with the same
# inference cost — you are buying generalisation for free at serving time.
#
# LEVER 1 — KAIMING (He) INITIALISATION:
#   Analogy: a relay race where every runner starts at a different
#   random speed. If the first runner starts too slow, the signal dies
#   before the last runner; too fast, and it overshoots.
#
#   PyTorch's default init for Conv2d/Linear is uniform over
#   +/-1/sqrt(fan_in) — derived for linear networks. ReLU kills half
#   the activations on average, shrinking the signal variance by ~2x
#   per layer. He et al. (2015) showed the right fix is to scale the
#   weight variance by 2/fan_in so the signal survives ReLU:
#       std = sqrt(2 / fan_in)
#   With BatchNorm the effect is partly masked (BN re-normalises), but
#   the first epochs still train measurably faster from a Kaiming start.
#
# LEVER 2 — LABEL SMOOTHING:
#   Analogy: a student who answers every true/false question with
#   "100% certain!" — even the ones they guessed. They learn nothing
#   from being wrong because every answer carries maximum confidence.
#
#   Hard targets tell the model the correct class has probability 1.0
#   and all others 0.0. Label smoothing (Szegedy et al. 2016) replaces
#   the targets with eps/(K-1) on the wrong classes and 1-eps on the
#   right one (here eps=0.1). The model is still rewarded for being
#   right, but is no longer rewarded for being ARBITRARILY confident —
#   softmax probabilities become honest estimates of uncertainty, which
#   is what a triage system needs (see Phase 5).
#
# LEVER 3 — MIXUP (vicinal risk minimisation):
#   Analogy: studying for an exam only with past papers teaches you the
#   past papers. Mixing two questions into a hybrid forces you to learn
#   the UNDERLYING concept, not the memorised answer.
#
#   Mixup (Zhang et al. 2018) trains on convex combinations of pairs:
#       x_mix = lam * x_i + (1 - lam) * x_j       lam ~ Beta(alpha, alpha)
#       loss  = lam * CE(f(x_mix), y_i) + (1 - lam) * CE(f(x_mix), y_j)
#   The model must behave LINEARLY between training examples, which
#   smooths the decision boundary and reduces memorisation. Caveat we
#   will observe honestly: Mixup regularises so aggressively that at a
#   fixed SHORT epoch budget it can trail the baseline — its benefit
#   shows with longer training. We report what we measure, not what the
#   paper's (much longer) runs report.

print("=" * 70)
print("  PHASE 1 — THEORY: Kaiming Init, Label Smoothing, Mixup")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: the levers and the ablation backbone


Conv/Linear weights: Normal(0, sqrt(2/fan_in)). Biases: zero.
    BatchNorm weights stay at 1 and biases at 0 (already the default).


Returns: x_mix, y_a, y_b, lam, perm — the mixed inputs, the two label
    sets, the mixing coefficient, and the permutation (returned so the
    mixing is auditable: x_mix == lam * x + (1 - lam) * x[perm]).


Convex combination of the two cross-entropies (same lam as inputs).


Two conv blocks (3->32->64, each BN+ReLU+pool) and a small head.
    Identical across variants so the ONLY difference is the lever.


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: kaiming_init + mixup_batch + AblationCNN")
print("=" * 70)

ABLATION_EPOCHS = 6
ABLATION_SUBSET = 20_000  # fixed prefix of the training tensors
SMOOTHING = 0.1
MIXUP_ALPHA = 0.2


def kaiming_init(model: nn.Module) -> nn.Module:
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            # TODO: Initialise m.weight with Kaiming normal — fan_in mode,
            #   ReLU gain — then zero m.bias when it exists
            # Hint: nn.init.kaiming_normal_(..., mode="fan_in", nonlinearity="relu")
            ____
            ____
    return model


def mixup_batch(
    x: torch.Tensor, y: torch.Tensor, alpha: float
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float, torch.Tensor]:
    # TODO: Sample lam from Beta(alpha, alpha) when alpha > 0, else 1.0
    # Hint: np.random.beta returns a numpy scalar — cast to float
    lam = ____
    # TODO: Build a random permutation of the batch indices (same device as x)
    perm = ____
    # TODO: The convex blend — lam * x + (1 - lam) * x[perm]
    x_mix = ____
    return x_mix, y, y[perm], lam, perm


def mixup_criterion(
    logits: torch.Tensor,
    y_a: torch.Tensor,
    y_b: torch.Tensor,
    lam: float,
    smoothing: float = 0.0,
) -> torch.Tensor:
    # TODO: lam * CE(logits, y_a) + (1 - lam) * CE(logits, y_b) — pass
    #   label_smoothing=smoothing to both
    return ____


class AblationCNN(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16 -> 8
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))



### Checkpoint 1: the levers are correct


In [ ]:
# (a) Kaiming std matches sqrt(2/fan_in) on a conv layer
torch.manual_seed(0)
_probe = AblationCNN()
_conv = _probe.features[0]
_default_std = float(_conv.weight.std().item())
kaiming_init(_probe)
_expected_std = float(np.sqrt(2.0 / (3 * 3 * 3)))  # fan_in = k*k*c_in = 27
_kaiming_std = float(_conv.weight.std().item())
assert abs(_kaiming_std - _expected_std) / _expected_std < 0.10, (
    f"Kaiming std {_kaiming_std:.4f} should be ~{_expected_std:.4f} "
    "(sqrt(2/fan_in), fan_in=27)"
)
assert float(_conv.bias.abs().max().item()) == 0.0, "Kaiming init zeroes biases"

# (b) Mixup produces exact convex combinations and preserves shapes
torch.manual_seed(0)
np.random.seed(0)
_x = torch.randn(8, 3, 32, 32)
_y = torch.arange(8)
_x_mix, _y_a, _y_b, _lam, _perm = mixup_batch(_x, _y, MIXUP_ALPHA)
assert _x_mix.shape == _x.shape, "Mixup must preserve input shape"
assert 0.0 <= _lam <= 1.0, f"lam={_lam} outside [0, 1]"
assert torch.equal(_y_a, _y) and torch.equal(_y_b, _y[_perm])
assert torch.allclose(_x_mix, _lam * _x + (1 - _lam) * _x[_perm], atol=1e-6), (
    "x_mix must be the exact convex combination lam*x + (1-lam)*x[perm]"
)
# (c) mixup_criterion reduces to plain CE when lam = 1
_logits = torch.randn(8, N_CLASSES)
_ce = F.cross_entropy(_logits, _y)
_mc = mixup_criterion(_logits, _y_a, _y_b, lam=1.0)
assert abs(float(_mc - _ce)) < 1e-6, "lam=1 must reduce to plain cross-entropy"

print(f"\nAblationCNN built: {count_parameters(_probe):,} parameters")
print(f"  default conv std={_default_std:.4f} -> kaiming std={_kaiming_std:.4f} "
      f"(target {_expected_std:.4f})")
print(f"  mixup audit: lam={_lam:.3f}, exact convex combination verified")
print("\n--- Checkpoint 1 passed --- levers verified\n")
del _probe, _x, _y, _x_mix, _y_a, _y_b, _perm, _logits



## PHASE 3 — TRAIN: four-variant ablation, one lever at a time


A plain torch loop is used instead of the shared Lightning harness
    because Mixup changes the per-batch loss computation (two label sets,
    convex weighting), which the shared LitCNN wrapper does not express.


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: ablation on CIFAR-10 (20K subset, full 10K val)")
print("=" * 70)

X_train, y_train, X_val, y_val, _, _ = load_cifar10()

# Fixed subset for the ablation — disclosed in the header. The SAME
# subset feeds every variant, so differences come from the levers, not
# from data sampling.
X_sub = X_train[:ABLATION_SUBSET]
y_sub = y_train[:ABLATION_SUBSET]
sub_loader = DataLoader(
    TensorDataset(X_sub, y_sub), batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=512)
print(
    f"Ablation data: {len(X_sub):,} train images (subset), "
    f"{len(X_val):,} val images (full)"
)

conn, tracker, exp_name, registry, has_registry = init_engines()


async def train_variant(
    name: str,
    use_kaiming: bool,
    smoothing: float,
    mixup_alpha: float,
) -> tuple[nn.Module, list[float], list[float]]:
    torch.manual_seed(42)  # identical init draw for every variant
    model = AblationCNN().to(DEVICE)
    if use_kaiming:
        kaiming_init(model)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    train_losses: list[float] = []
    val_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "variant": name,
                "kaiming_init": str(use_kaiming),
                "label_smoothing": str(smoothing),
                "mixup_alpha": str(mixup_alpha),
                "epochs": str(ABLATION_EPOCHS),
                "subset_size": str(ABLATION_SUBSET),
                "architecture": "AblationCNN",
            }
        )
        for epoch in range(ABLATION_EPOCHS):
            model.train()
            batch_losses = []
            for xb, yb in sub_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                if mixup_alpha > 0:
                    # TODO: Mix the batch, forward, and take the mixup loss
                    #   1. mixup_batch gives xb, y_a, y_b, lam (ignore perm)
                    #   2. loss = mixup_criterion(model(xb), y_a, y_b, lam, smoothing)
                    xb, y_a, y_b, lam, _ = ____
                    logits = model(xb)
                    loss = ____
                else:
                    # TODO: Plain (optionally smoothed) cross-entropy
                    # Hint: F.cross_entropy(..., label_smoothing=smoothing)
                    logits = model(xb)
                    loss = ____
                opt.zero_grad()
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())

            model.eval()
            correct = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    correct += int((model(xb).argmax(-1) == yb).sum().item())
            val_acc = correct / len(y_val)

            train_losses.append(float(np.mean(batch_losses)))
            val_accs.append(val_acc)
            await run.log_metrics(
                {"train_loss": train_losses[-1], "val_accuracy": val_acc},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1}/{ABLATION_EPOCHS}  "
                f"train={train_losses[-1]:.4f}  val={val_acc:.3f}"
            )
        await run.log_metrics(
            {
                "final_train_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
            }
        )
    return model, train_losses, val_accs


VARIANTS = [
    ("baseline", False, 0.0, 0.0),
    ("kaiming", True, 0.0, 0.0),
    ("kaiming+smoothing", True, SMOOTHING, 0.0),
    ("kaiming+smoothing+mixup", True, SMOOTHING, MIXUP_ALPHA),
]

results: dict[str, dict[str, object]] = {}
for name, use_k, smooth, alpha in VARIANTS:
    print(f"\nTraining variant: {name}")
    model_v, losses_v, accs_v = asyncio.run(
        train_variant(name, use_k, smooth, alpha)
    )
    results[name] = {"model": model_v, "losses": losses_v, "accs": accs_v}



### Checkpoint 2: all variants trained and converging


In [ ]:
for name, res in results.items():
    losses = res["losses"]
    accs = res["accs"]
    assert len(losses) == ABLATION_EPOCHS, f"{name}: expected {ABLATION_EPOCHS} epochs"
    assert losses[-1] < losses[0], f"{name}: train loss should decrease"
    assert accs[-1] > 0.30, (
        f"{name}: val acc {accs[-1]:.3f} too low — expected > 0.30 after "
        f"{ABLATION_EPOCHS} epochs on the 20K subset"
    )

print(f"\n{'=' * 62}")
print("  ABLATION RESULTS (measured, this run)")
print(f"{'=' * 62}")
print(f"  {'Variant':>26} {'Final Loss':>12} {'Val Acc':>10}")
print("  " + "-" * 56)
for name, res in results.items():
    print(
        f"  {name:>26} {res['losses'][-1]:>12.4f} {res['accs'][-1]:>9.3f}"
    )

base_acc = results["baseline"]["accs"][-1]
mix_acc = results["kaiming+smoothing+mixup"]["accs"][-1]
print(
    f"\n  Mixup delta at {ABLATION_EPOCHS} epochs: {mix_acc - base_acc:+.3f}. "
    "Mixup regularises aggressively — at short budgets it can trail the "
    "baseline; its benefit shows with longer training. That is the honest "
    "reading of THIS run, and it is also what Zhang et al. report: mixup "
    "wins appear at full training length, not in the first epochs."
)
print("\n--- Checkpoint 2 passed --- ablation trained and tracked\n")

# Register the best-val-accuracy variant
best_name = max(results, key=lambda n: results[n]["accs"][-1])
best = results[best_name]
if has_registry:
    register_model(
        registry,
        "ablation_cifar10",
        best["model"],
        best["losses"][-1],
        best["accs"][-1],
        epochs=ABLATION_EPOCHS,
    )
    print(f"  Best variant registered: {best_name}")



## PHASE 4 — VISUALISE: curves, Mixup composites, and the confidence shift


Return (mean max-prob, accuracy) over the validation set.


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: what each lever actually did")
print("=" * 70)

# ModelVisualizer carries a kailash-ml P2 experimental notice (a UserWarning
# at construction). The gate runs with warnings-as-errors, so acknowledge it
# narrowly here — only the constructor notice is muted, everything else
# still errors.
from kailash_ml._decorators import ExperimentalWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ExperimentalWarning)
    viz = create_visualizer()
save_training_plots(
    viz,
    {name: res["accs"] for name, res in results.items()},
    "ex_2_05_ablation_val_accuracy.html",
    y_label="Validation Accuracy",
)
save_training_plots(
    viz,
    {name: res["losses"] for name, res in results.items()},
    "ex_2_05_ablation_train_loss.html",
    y_label="Training Loss",
)

# (a) Mixup composites — the visual proof of "training between examples"
torch.manual_seed(7)
np.random.seed(7)
_pair_x = X_val[:8]
_pair_y = y_val[:8]
_mixed, _ya, _yb, _lam, _perm = mixup_batch(_pair_x, _pair_y, 0.5)

fig_mix, axes = plt.subplots(3, 8, figsize=(20, 8))
fig_mix.suptitle(
    f"Mixup composites (lam={_lam:.2f}): image A, image B, and the blend the model trains on",
    fontsize=13,
)
for col in range(8):
    a = denormalise_cifar(_pair_x[col]).permute(1, 2, 0).numpy()
    b = denormalise_cifar(_pair_x[_perm[col]]).permute(1, 2, 0).numpy()
    m = denormalise_cifar(_mixed[col]).permute(1, 2, 0).numpy()
    axes[0, col].imshow(a)
    axes[0, col].set_title(f"A: {CLASS_NAMES[_ya[col]]}", fontsize=8)
    axes[1, col].imshow(b)
    axes[1, col].set_title(f"B: {CLASS_NAMES[_yb[col]]}", fontsize=8)
    axes[2, col].imshow(m)
    axes[2, col].set_title("blend", fontsize=8)
    for row in range(3):
        axes[row, col].axis("off")
axes[0, 0].set_ylabel("image A", fontsize=10)
axes[1, 0].set_ylabel("image B", fontsize=10)
axes[2, 0].set_ylabel("lam*A+(1-lam)*B", fontsize=10)
plt.tight_layout()
plt.savefig("ex_2_05_mixup_composites.png", dpi=150, bbox_inches="tight")
plt.close(fig_mix)
print("  Saved: ex_2_05_mixup_composites.png")

# (b) Confidence shift — label smoothing's signature.
# Measure the mean max-softmax probability on the val set for the
# baseline (hard targets) vs the smoothed variant. Overconfident models
# sit near 1.0 even when wrong; smoothed models keep calibrated margins.
def mean_confidence(model: nn.Module) -> tuple[float, float]:
    model.eval()
    probs, preds = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            # TODO: Softmax the logits, then collect the MAX probability
            #   (confidence) and the argmax (prediction) per image
            p = ____
            probs.append(____)
            preds.append(____)
    probs_t = torch.cat(probs)
    preds_t = torch.cat(preds)
    acc = float((preds_t == y_val).float().mean().item())
    return float(probs_t.mean().item()), acc


conf_base, acc_base = mean_confidence(results["baseline"]["model"])
conf_smooth, acc_smooth = mean_confidence(results["kaiming+smoothing"]["model"])
print(
    f"\n  Mean max-softmax confidence on 10K val images:\n"
    f"    baseline (hard targets):   {conf_base:.3f}  (acc {acc_base:.3f})\n"
    f"    kaiming+smoothing:         {conf_smooth:.3f}  (acc {acc_smooth:.3f})"
)

fig_conf, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    ["baseline", "kaiming+smoothing"],
    [conf_base, conf_smooth],
    color=["#d62728", "#2ca02c"],
)
ax.set_ylabel("Mean max-softmax probability")
ax.set_ylim(0, 1.0)
ax.axhline(acc_base, color="#d62728", linestyle="--", alpha=0.6,
           label=f"baseline acc ({acc_base:.2f})")
ax.axhline(acc_smooth, color="#2ca02c", linestyle="--", alpha=0.6,
           label=f"smoothed acc ({acc_smooth:.2f})")
ax.set_title("Label smoothing closes the confidence-accuracy gap")
ax.legend(fontsize=9)
fig_conf.tight_layout()
fig_conf.savefig("ex_2_05_confidence_shift.png", dpi=150, bbox_inches="tight")
plt.close(fig_conf)
print("  Saved: ex_2_05_confidence_shift.png")



### Checkpoint 3: smoothing measurably reduces overconfidence


In [ ]:
# Allow slack for run-to-run noise, but the direction must hold: a
# smoothed model must not be MORE overconfident than the baseline.
assert conf_smooth < conf_base + 0.05, (
    f"Smoothed confidence {conf_smooth:.3f} should not exceed baseline "
    f"{conf_base:.3f} by more than noise (0.05)"
)
import os

for artefact in (
    "ex_2_05_ablation_val_accuracy.html",
    "ex_2_05_mixup_composites.png",
    "ex_2_05_confidence_shift.png",
):
    assert os.path.exists(artefact), f"Missing artefact: {artefact}"
print("\n--- Checkpoint 3 passed --- visual proof generated\n")



## PHASE 5 — APPLY: crop-disease triage with limited labelled data


Auto-accept predictions above threshold; return pool statistics.


TRIAGE POLICY: auto-route photos with confidence >= {THRESHOLD:.2f}

  {'Model':>24} {'Auto-routed':>14} {'Accuracy of auto-routed':>26}
  {'-' * 66}
  {'baseline (hard targets)':>24} {rep_base['accept_rate']:>13.1%} {rep_base['accepted_acc']:>25.1%}
  {'kaiming+smoothing':>24} {rep_smooth['accept_rate']:>13.1%} {rep_smooth['accepted_acc']:>25.1%}

  READING IT:
    The baseline auto-routes a larger share, but its auto-routed pool is
    less trustworthy — hard targets teach the model to say "95%" on
    photos it should be unsure about. The smoothed model's threshold
    means what it says. For a farmer waiting on a spray/no-spray call,
    the second row is the one an agronomist can defend.

  STAKEHOLDER-READY OUTPUT:
    "At a 0.90 confidence threshold the baseline auto-routes
    {rep_base['accept_rate']:.0%} of photos at {rep_base['accepted_acc']:.0%}
    accuracy; the smoothed model auto-routes {rep_smooth['accept_rate']:.0%}
    at {rep_smooth['accepted_acc']:.0%}. Smoothing trades a little
    automation volume for a triage queue whose confidence numbers can be
    audited — and Mixup buys extra generalisation from the same 20K
    agronomist labels when the training budget allows longer runs."


In [ ]:
# SCENARIO (anonymised, illustrative): a regional agritech startup runs
# drone-photo triage for smallholder farms. Agronomists labelled 20,000
# field photos into 10 crop-condition classes — exactly our ablation
# setup. Two business constraints make the levers matter:
#
#   1. LABELS ARE EXPENSIVE: every new labelled photo costs an
#      agronomist's time. Regularisation (Mixup) squeezes more
#      generalisation out of the same 20K labels.
#   2. TRIAGE DECISIONS ARE THRESHOLDED: photos the model is confident
#      about are auto-routed; the rest go to a human queue. If the model
#      says "95% confident" but is right only 70% of the time at that
#      confidence, the auto-route queue silently ships errors to farmers.
#      Label smoothing makes the confidence number MEAN something.
#
# We simulate the triage policy on the val set: auto-accept predictions
# whose confidence exceeds a threshold, measure the accuracy of the
# auto-accepted pool for the baseline vs the smoothed model.

print("=" * 70)
print("  PHASE 5 — APPLY: confidence-thresholded crop-disease triage")
print("=" * 70)


def triage_report(model: nn.Module, threshold: float) -> dict[str, float]:
    model.eval()
    confs, preds = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            p = F.softmax(model(xb.to(DEVICE)), dim=-1)
            confs.append(p.max(dim=-1).values.cpu())
            preds.append(p.argmax(-1).cpu())
    conf_t = torch.cat(confs)
    pred_t = torch.cat(preds)
    # TODO: Boolean mask of predictions whose confidence clears the threshold
    accepted = ____
    n_acc = int(accepted.sum().item())
    if n_acc == 0:
        return {"accept_rate": 0.0, "accepted_acc": 0.0}
    # TODO: Accuracy computed ONLY over the auto-accepted pool
    # Hint: boolean-index pred_t and y_val with the mask
    acc_accepted = ____
    return {"accept_rate": n_acc / len(y_val), "accepted_acc": acc_accepted}


THRESHOLD = 0.90
rep_base = triage_report(results["baseline"]["model"], THRESHOLD)
rep_smooth = triage_report(results["kaiming+smoothing"]["model"], THRESHOLD)

print(
)



### Checkpoint 4: triage simulation computed on the full val set


In [ ]:
assert rep_base["accept_rate"] > 0.0, "Baseline should auto-route some photos"
assert rep_smooth["accepted_acc"] > 0.5, (
    f"Smoothed auto-routed accuracy {rep_smooth['accepted_acc']:.3f} too low"
)
print("--- Checkpoint 4 passed --- triage application demonstrated\n")



## Clean up


In [ ]:
await conn.close()



## REFLECTION


THEORY:
  [x] Kaiming init: std = sqrt(2/fan_in) keeps signal variance alive
      through ReLU stacks (measured: {_kaiming_std:.4f} vs target
      {_expected_std:.4f} on the first conv)
  [x] Label smoothing: soft targets (0.9/0.01) stop rewarding arbitrary
      confidence (measured: {conf_base:.3f} -> {conf_smooth:.3f} mean
      max-probability)
  [x] Mixup: convex blends of image PAIRS force linear behaviour between
      examples — verified exact: x_mix == lam*x + (1-lam)*x[perm]

  BUILD + TRAIN:
  [x] kaiming_init / mixup_batch / mixup_criterion as reusable levers
  [x] 4-variant ablation on a fixed 20K CIFAR-10 subset, one lever at a
      time, every run tracked in ExperimentTracker
  [x] Honest reading: at {ABLATION_EPOCHS} epochs Mixup's delta is
      {mix_acc - base_acc:+.3f} — regularisation pays off with longer
      budgets, and we report the measured sign, not the hoped-for one

  VISUALISE (the proof):
  [x] Ablation val-accuracy and loss curves (all four variants overlaid)
  [x] Mixup composites: image A, image B, and the exact blend trained on
  [x] Confidence shift: smoothed models close the confidence-accuracy gap

  APPLY:
  [x] Crop-disease triage with a 0.90 confidence threshold
  [x] Baseline auto-routes {rep_base['accept_rate']:.0%} at
      {rep_base['accepted_acc']:.0%} accuracy; smoothed auto-routes
      {rep_smooth['accept_rate']:.0%} at {rep_smooth['accepted_acc']:.0%}
  [x] The business lever is TRUST IN THE THRESHOLD, not raw accuracy

  KEY INSIGHT: Training-time levers are free at inference — but only
  measured levers earn their place. Kaiming initialisation, label
  smoothing and Mixup each move a different dial (optimisation,
  calibration, regularisation); the ablation discipline — one lever at
  a time, same data, same seed, tracked runs — is what turns folklore
  into an engineering decision.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

